In [1]:
import numpy as np
import os
import re

base_path = r"Data\Data_All_The_BDH_PostProcess"

for alpha_folder in sorted(os.listdir(base_path)):
    alpha_path = os.path.join(base_path, alpha_folder)
    if not os.path.isdir(alpha_path):
        continue

    for building_folder in sorted(os.listdir(alpha_path)):
        building_path = os.path.join(alpha_path, building_folder)
        data_path = os.path.join(building_path, "Data")
        if not os.path.isdir(data_path):
            continue

        # Detect available angles from windward files
        angles = sorted({
            int(m.group(1))
            for f in os.listdir(data_path)
            if (m := re.match(r"windward_avg_angle_(\d+)\.npy$", f))
        })

        print(f"\n--- {alpha_folder}/{building_folder} (angles: {angles[0]}-{angles[-1]}) ---")
        for angle in angles:
            windward_file = os.path.join(data_path, f"windward_avg_angle_{angle}.npy")
            leeward_file = os.path.join(data_path, f"leeward_avg_angle_{angle}.npy")

            if not os.path.exists(windward_file) or not os.path.exists(leeward_file):
                print(f"  Angle {angle:>3}: MISSING files, skipped")
                continue

            windward = np.load(windward_file)
            leeward = np.load(leeward_file)
            drag = windward - leeward

            # Save drag array
            drag_file = os.path.join(data_path, f"drag_avg_angle_{angle}.npy")
            np.save(drag_file, drag)
            print(f"  Angle {angle:>3}: drag mean={drag.mean():.4f}, std={drag.std():.4f} -> saved {drag_file}")

print("\nDone! All drag files saved.")


--- Alpha1_4/1_1_2 (angles: 0-50) ---
  Angle   0: drag mean=0.9407, std=0.2859 -> saved Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Data\drag_avg_angle_0.npy
  Angle   5: drag mean=0.9830, std=0.2859 -> saved Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Data\drag_avg_angle_5.npy
  Angle  10: drag mean=0.9902, std=0.2956 -> saved Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Data\drag_avg_angle_10.npy
  Angle  15: drag mean=0.9739, std=0.2913 -> saved Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Data\drag_avg_angle_15.npy
  Angle  20: drag mean=0.9761, std=0.2860 -> saved Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Data\drag_avg_angle_20.npy
  Angle  25: drag mean=0.9693, std=0.2655 -> saved Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Data\drag_avg_angle_25.npy
  Angle  30: drag mean=0.9505, std=0.2706 -> saved Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Data\drag_avg_angle_30.npy
  Angle  35: drag mean=0.9550, std=0.2769 -> saved Data\Data_All_The_BDH_P

In [3]:
import numpy as np
import os
import re
import matplotlib.pyplot as plt

base_path = r"Data\Data_All_The_BDH_PostProcess"

def plot_drag(drag, angle, alpha_folder, building_folder, images_path):
    avg_diff = drag.mean()
    std_diff = drag.std()

    plt.figure(figsize=(12, 6))
    plt.plot(drag, label='Windward - Leeward', color='purple', linewidth=1.5)
    plt.axhline(y=avg_diff, color='red', linestyle='--', linewidth=1.5,
                label=f'Average: {avg_diff:.4f}')
    plt.fill_between(
        range(len(drag)),
        drag - std_diff,
        drag + std_diff,
        color='purple', alpha=0.2,
        label=f'Standard Deviation: ±{std_diff:.4f}'
    )
    plt.title(f'Drag  —  {alpha_folder} / {building_folder}  (Angle {angle}°)',
              fontsize=14, fontweight='bold')
    plt.xlabel('Sample Index', fontsize=12)
    plt.ylabel('Pressure Coefficient $C_p$', fontsize=12)
    plt.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.7)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    out_file = os.path.join(images_path, f"drag_angle_{angle}.png")
    plt.savefig(out_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {out_file}")


for alpha_folder in sorted(os.listdir(base_path)):
    alpha_path = os.path.join(base_path, alpha_folder)
    if not os.path.isdir(alpha_path):
        continue

    for building_folder in sorted(os.listdir(alpha_path)):
        building_path = os.path.join(alpha_path, building_folder)
        data_path = os.path.join(building_path, "Data")
        images_path = os.path.join(building_path, "Images")
        if not os.path.isdir(data_path):
            continue
        os.makedirs(images_path, exist_ok=True)

        # Detect available angles from drag files
        angles = sorted({
            int(m.group(1))
            for f in os.listdir(data_path)
            if (m := re.match(r"drag_avg_angle_(\d+)\.npy$", f))
        })

        if not angles:
            print(f"No drag files found for {alpha_folder}/{building_folder}, skipping.")
            continue

        print(f"\n--- {alpha_folder}/{building_folder} ---")
        for angle in angles:
            drag_file = os.path.join(data_path, f"drag_avg_angle_{angle}.npy")
            drag = np.load(drag_file)
            plot_drag(drag, angle, alpha_folder, building_folder, images_path)

print("\nDone! All drag plots saved.")


--- Alpha1_4/1_1_2 ---
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_0.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_5.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_10.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_15.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_20.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_25.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_30.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_35.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_40.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_45.png
  ✓ Saved: Data\Data_All_The_BDH_PostProcess\Alpha1_4\1_1_2\Images\drag_angle_50.png

--- Alpha1_4/1_1_3 ---
  ✓ Saved: Data\Dat